In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import gymnasium as gym
import numpy as np
import random
import matplotlib.pyplot as plt

In [2]:
# 1. Simple Neural Network to approximate Q-values
class QNetwork(nn.Module):
    def __init__(self, state_dim, action_dim):
        super(QNetwork, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(state_dim, 64),
            nn.ReLU(),
            nn.Linear(64, action_dim)
        )

    def forward(self, x):
        return self.fc(x)

In [3]:
from collections import deque
# 2. Replay Buffer (Standard)
class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)
    def push(self, *args): 
        self.buffer.append(args)
    def sample(self, batch_size): 
        return random.sample(self.buffer, batch_size)
    def __len__(self):
        return len(self.buffer)

In [ ]:
import gymnasium as gym
import highway_env

def train(episodes=500):
    env = gym.make('highway-v0')
    state_dim = np.prod(env.observation_space.shape)
    action_dim = env.action_space.n
    
    q_net = QNetwork(state_dim, action_dim)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    q_net = q_net.to(device)

    print(f"Training on: {device}")
    print(f"GPU name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")

    optimizer = optim.Adam(q_net.parameters(), lr=0.001)
    loss_fn = nn.MSELoss()

    # Hyperparameters
    gamma = 0.99
    epsilon = 1.0
    epsilon_decay = 0.995
    episodes = episodes

    for ep in range(episodes):
        state, _ = env.reset()
        done = False

        total_loss = 0
        total_reward = 0
        
        while not done:
            state_tensor = torch.FloatTensor(state.flatten()).to(device)
            
            # Epsilon-greedy action selection
            if random.random() < epsilon:
                action = env.action_space.sample()
            else:
                with torch.no_grad():
                    action = q_net(state_tensor).argmax().item()

            next_state, reward, terminated, truncated, _ = env.step(action)
            total_reward += reward
            done = terminated or truncated

            # Prepare Target 
            next_state_tensor = torch.FloatTensor(next_state.flatten()).to(device)
            
            with torch.no_grad():
                next_q_values = q_net(next_state_tensor)
                max_next_q = torch.max(next_q_values)
                # Bellman Target: r + gamma * max(Q(s', a'))
                target_q = reward + (gamma * max_next_q * (1 - terminated))
                target_q = target_q.to(device)

            # Current Q-value prediction
            current_q = q_net(state_tensor)[action]

            # Backpropagation
            loss = loss_fn(current_q, target_q)
            optimizer.zero_grad()
            loss.backward()
            total_loss += loss.item()
            optimizer.step()

            state = next_state

        epsilon = max(0.01, epsilon * epsilon_decay)
        
        
        print(f"Episode {ep+1} | Reward: {total_reward:.2f} | Loss: {total_loss:.4f} | Epsilon: {epsilon:.2f}")

    env.close()
    return q_net


In [ ]:
train()

## 1st Improvement: Target Q-Network

In [ ]:

import gymnasium as gym
import highway_env


def train_target_qnetwork(episodes=500):
    env = gym.make('highway-v0')
    state_dim = np.prod(env.observation_space.shape)
    action_dim = env.action_space.n

    policy_net = QNetwork(state_dim, action_dim)
    target_net = QNetwork(state_dim, action_dim)
    target_net.load_state_dict(policy_net.state_dict())

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    policy_net = policy_net.to(device)
    target_net = target_net.to(device)

    print(f"Training on: {device}")
    print(f"GPU name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")

    optimizer = optim.Adam(policy_net.parameters(), lr=0.001)
    loss_fn = nn.MSELoss()

    # Hyperparameters
    gamma = 0.99
    epsilon = 1.0
    epsilon_decay = 0.995
    episodes = episodes

    target_update_freq = 10

    for ep in range(episodes):
        total_loss = 0
        total_reward = 0
        state, _ = env.reset()
        done = False
        step_count = 0
        
        while not done:
            step_count += 1
            state_tensor = torch.FloatTensor(state.flatten()).to(device)
            
            # Epsilon-greedy action selection
            if random.random() < epsilon:
                action = env.action_space.sample()
            else:
                with torch.no_grad():
                    action = policy_net(state_tensor).argmax().item()

            next_state, reward, terminated, truncated, _ = env.step(action)
            total_reward += reward
            done = terminated or truncated

            # Prepare Target 
            next_state_tensor = torch.FloatTensor(next_state.flatten()).to(device)
            
            with torch.no_grad():
                next_q_values = target_net(next_state_tensor)
                max_next_q = torch.max(next_q_values)
                # Bellman Target: r + gamma * max(Q(s', a'))
                target_q = reward + (gamma * max_next_q * (1 - terminated))
                target_q = target_q.to(device)

            # Current Q-value prediction
            current_q = policy_net(state_tensor)[action]

            # Backpropagation
            loss = loss_fn(current_q, target_q)
            optimizer.zero_grad()
            loss.backward()
            total_loss += loss.item()
            optimizer.step()

            state = next_state

            avg_reward = total_reward / step_count
            avg_loss = total_loss / step_count

        if ep % target_update_freq == 0:
            target_net.load_state_dict(policy_net.state_dict())

        epsilon = max(0.01, epsilon * epsilon_decay)
        
        
        print(f"Episode {ep+1} | Average Reward: {avg_reward:.2f} | Average Loss: {avg_loss:.4f} | Epsilon: {epsilon:.2f}")

    env.close()
    return policy_net


In [6]:
import gymnasium as gym
import highway_env

def train_double_dqn(episodes=500):
    env = gym.make('highway-v0')
    state_dim = np.prod(env.observation_space.shape)
    action_dim = env.action_space.n

    policy_net = QNetwork(state_dim, action_dim)
    target_net = QNetwork(state_dim, action_dim)
    target_net.load_state_dict(policy_net.state_dict())

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    policy_net = policy_net.to(device)
    target_net = target_net.to(device)

    print(f"Training on: {device}")
    print(f"GPU name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")

    optimizer = optim.Adam(policy_net.parameters(), lr=0.001)
    loss_fn = nn.MSELoss()

    gamma = 0.99
    epsilon = 1.0
    epsilon_decay = 0.995
    min_epsilon = 0.01
    target_update_freq = 2

    rewards_history = []
    loss_history = []

    for ep in range(episodes):
        state, _ = env.reset()
        done = False

        total_loss = 0
        total_reward = 0
        step_count = 0

        while not done:
            step_count += 1
            state_tensor = torch.tensor(state.flatten(), dtype=torch.float32, device=device)

            if random.random() < epsilon:
                action = env.action_space.sample()
            else:
                with torch.no_grad():
                    action = policy_net(state_tensor).argmax().item()

            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            total_reward += reward

            next_state_tensor = torch.tensor(next_state.flatten(), dtype=torch.float32, device=device)

            with torch.no_grad():
                next_action = policy_net(next_state_tensor).argmax()
                next_q_value = target_net(next_state_tensor)[next_action]
                target_q = torch.tensor(reward, dtype=torch.float32, device=device) + gamma * next_q_value * (1 - int(done))

            current_q = policy_net(state_tensor)[action]

            loss = loss_fn(current_q, target_q)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            state = next_state

        if ep % target_update_freq == 0:
            target_net.load_state_dict(policy_net.state_dict())

        epsilon = max(min_epsilon, epsilon * epsilon_decay)
        avg_loss = total_loss / max(step_count, 1)

        rewards_history.append(total_reward)
        loss_history.append(avg_loss)

        print(
            f"Episode {ep + 1} | "
            f"Reward: {total_reward:.2f} | "
            f"Average Loss: {avg_loss:.4f} | "
            f"Epsilon: {epsilon:.2f}"
        )

    env.close()
    return policy_net, rewards_history, loss_history

In [7]:
double_dqn_model, double_dqn_rewards, double_dqn_losses = train_double_dqn(episodes=500)

Training on: cpu
GPU name: N/A
Episode 1 | Reward: 16.36 | Average Loss: 0.8183 | Epsilon: 0.99
Episode 2 | Reward: 6.95 | Average Loss: 0.9702 | Epsilon: 0.99
Episode 3 | Reward: 5.00 | Average Loss: 0.7376 | Epsilon: 0.99
Episode 4 | Reward: 1.95 | Average Loss: 0.7833 | Epsilon: 0.98
Episode 5 | Reward: 19.47 | Average Loss: 0.7198 | Epsilon: 0.98
Episode 6 | Reward: 5.27 | Average Loss: 0.8746 | Epsilon: 0.97
Episode 7 | Reward: 27.83 | Average Loss: 0.5737 | Epsilon: 0.97
Episode 8 | Reward: 29.38 | Average Loss: 0.5547 | Epsilon: 0.96
Episode 9 | Reward: 4.39 | Average Loss: 0.8309 | Epsilon: 0.96
Episode 10 | Reward: 12.89 | Average Loss: 1.2760 | Epsilon: 0.95
Episode 11 | Reward: 5.11 | Average Loss: 1.1685 | Epsilon: 0.95
Episode 12 | Reward: 5.79 | Average Loss: 1.4719 | Epsilon: 0.94
Episode 13 | Reward: 14.94 | Average Loss: 0.9717 | Epsilon: 0.94
Episode 14 | Reward: 6.78 | Average Loss: 1.6966 | Epsilon: 0.93
Episode 15 | Reward: 3.80 | Average Loss: 3.6312 | Epsilon: 0.

KeyboardInterrupt: 

In [ ]:
def training_results(results, title='DQN Training Rewards'):
    plt.figure(figsize=(10, 5))

    for label, rewards in results.items():
        plt.plot(rewards, label=label)

    plt.xlabel('Episode')
    plt.ylabel('Total Reward')
    plt.title(title)
    plt.legend()
    plt.grid(True)
    plt.show()